# 02 — Train the TITANS memory mixer
Loads the immutable Transformer preflight contract, consumes the identical ordered token schedule, and evaluates adaptive memory against a segment-reset ablation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ATCG_COMMIT = '59cda3673f0cf9ab2720ea8d47fd47605ecaa865'
!test -d /content/ATCG-FM || git clone https://github.com/DRAGGON-Lab/ATCG-FM.git /content/ATCG-FM
!git -C /content/ATCG-FM fetch origin {ATCG_COMMIT}
!git -C /content/ATCG-FM checkout --detach {ATCG_COMMIT}
%pip install -q -e /content/ATCG-FM/packages/atcg-sequence --no-deps -e /content/ATCG-FM/packages/atcg-models --no-deps -e /content/ATCG-FM/packages/atcg-runtime --no-deps


In [ ]:
from pathlib import Path
from dataclasses import asdict
import gzip, hashlib, json, shutil, torch
from atcg.models import GenomicLanguageModel, titans_memory_tiny
from atcg.runtime import CudaUtilizationSampler, OrderedComparisonConfig, OrderedComparisonTrainer, ordered_dataset_fingerprint, restore_ordered_trainer, save_checkpoint, validate_ordered_model, validate_repeat_contexts
from atcg.sequence import FixedAlphabetTokenizer, OrderedCausalStreamDataset, parse_fasta

if not torch.cuda.is_available() or 'T4' not in torch.cuda.get_device_properties(0).name: raise RuntimeError('Select an NVIDIA T4 runtime')
ROOT = Path('/content/drive/MyDrive/ATCG-FM/ecoli_hybrid_v1'); STAGE_NAME = 'stage-small'; STAGE = ROOT / STAGE_NAME
SPLITS = ('train','validation_within','test_within','validation_clade','test_clade'); EVALUATION_SPLITS = SPLITS[1:]
experiment = json.loads((STAGE / 'experiment_config.json').read_text()); manifest = json.loads((STAGE / 'manifest.json').read_text())
if experiment['schema_version'] != 2 or experiment['stage'] != STAGE_NAME: raise RuntimeError('experiment schema/stage mismatch')
if experiment['dataset_fingerprint'] != manifest['dataset_fingerprint'] or experiment['split_sha256'] != manifest['split_sha256']: raise RuntimeError('dataset changed after Transformer preflight')
LOCAL = Path('/content/ecoli_hybrid_stage_small'); LOCAL.mkdir(exist_ok=True)
for split in SPLITS: shutil.copy2(STAGE / f'{split}.fa.gz', LOCAL / f'{split}.fa.gz')
for split in SPLITS:
    if hashlib.sha256((LOCAL / f'{split}.fa.gz').read_bytes()).hexdigest() != manifest['split_sha256'][split]: raise RuntimeError(f'{split} checksum mismatch')
def records(name):
    with gzip.open(LOCAL / f'{name}.fa.gz', 'rt') as handle: return parse_fasta(handle, source=str(LOCAL / f'{name}.fa.gz'))
tokenizer = FixedAlphabetTokenizer()
def dataset(values): return OrderedCausalStreamDataset(values, tokenizer, segment_length=experiment['segment_length'], gradient_horizon=experiment['gradient_horizon'], include_bos=True, include_eos=False)
train_records = records('train'); train_data = dataset(train_records); schedule_hash = ordered_dataset_fingerprint(train_data)
train_tokens = sum(len(segment.target_ids) for horizon in train_data for segment in horizon.segments)
if schedule_hash != experiment['schedule_hash'] or train_tokens != experiment['train_tokens']: raise RuntimeError('ordered token schedule differs from Transformer preflight')


In [ ]:
profile = experiment['profile']; torch.manual_seed(experiment['seed'])
model = GenomicLanguageModel(titans_memory_tiny(tokenizer.vocab_size, max_seq_len=experiment['segment_length'], d_model=profile['d_model'], n_layers=profile['layers'], expansion_factor=2, projection_kernel_size=4))
config = OrderedComparisonConfig(global_batch_size=experiment['global_batch_size'], microbatch_size=experiment['titans_microbatch'], learning_rate=experiment['learning_rate'], weight_decay=experiment['weight_decay'], precision='float16', device='cuda', seed=experiment['seed'])
trainer = OrderedComparisonTrainer(model, pad_id=tokenizer.pad_id, segment_length=experiment['segment_length'], config=config, memory_mode='adaptive')
run_dir = ROOT / 'runs' / STAGE_NAME / 'titans-memory-seed17'; run_dir.mkdir(parents=True, exist_ok=True)
resume_path = run_dir / 'resume.pt'; metrics_path = run_dir / 'metrics.jsonl'
metrics = [json.loads(line) for line in metrics_path.read_text().splitlines()] if resume_path.exists() and metrics_path.exists() else []
start_batch = restore_ordered_trainer(str(resume_path), trainer, dataset_fingerprint=experiment['dataset_fingerprint']) if resume_path.exists() else 0
torch.cuda.reset_peak_memory_stats()
with CudaUtilizationSampler() as gpu_sampler:
    for batch_index, horizons in enumerate(train_data.iter_batches(config.global_batch_size)):
        if batch_index < start_batch: continue
        row = trainer.train_global_batch(horizons); metrics.append(asdict(row))
        if (batch_index + 1) % 100 == 0:
            save_checkpoint(resume_path, model=model, optimizer=trainer.optimizer, training_state=trainer.state, stream_state=trainer.state_store.state_dict(), grad_scaler_state=trainer.scaler.state_dict(), experiment_state={'dataset_fingerprint':experiment['dataset_fingerprint'],'schedule_hash':schedule_hash,'global_batch_index':batch_index+1})
            with metrics_path.open('w') as handle:
                for value in metrics: handle.write(json.dumps(value, sort_keys=True) + '\n')
        if batch_index % 100 == 0: print(batch_index, metrics[-1])
wall_time = sum(value['elapsed_seconds'] for value in metrics); training_peak = torch.cuda.max_memory_allocated()
checkpoint = save_checkpoint(run_dir / 'last.pt', model=model, optimizer=trainer.optimizer, training_state=trainer.state, stream_state=trainer.state_store.state_dict(), grad_scaler_state=trainer.scaler.state_dict(), experiment_state={'dataset_fingerprint':experiment['dataset_fingerprint'],'schedule_hash':schedule_hash,'global_batch_index':len(metrics)})
with metrics_path.open('w') as handle:
    for value in metrics: handle.write(json.dumps(value, sort_keys=True) + '\n')
resume_path.unlink(missing_ok=True)
if trainer.state.tokens_seen != experiment['train_tokens']: raise RuntimeError('training did not consume exactly one shared token pass')


In [ ]:
evaluation_records = {name:records(name) for name in EVALUATION_SPLITS}
evaluation = {}
for policy, reset in [('adaptive',False),('reset_each_segment',True)]:
    evaluation[policy] = {name:asdict(validate_ordered_model(model, dataset(values), pad_id=tokenizer.pad_id, batch_size=config.microbatch_size, device='cuda', memory_mode='adaptive', offset_boundaries=(128,1024,4096,16384), reset_state_each_segment=reset)) for name,values in evaluation_records.items()}
def stream_scores(name):
    output = []
    for record in evaluation_records[name]:
        score = validate_ordered_model(model, dataset([record]), pad_id=tokenizer.pad_id, batch_size=1, device='cuda', memory_mode='adaptive')
        output.append({'stream_id':record.identifier,'accession':record.identifier.split('__',1)[0],'bits_per_token':score.bits_per_token,'token_count':score.token_count})
    return output
stream_evaluation = {name:stream_scores(name) for name in ('test_within','test_clade')}
repeat_context = asdict(validate_repeat_contexts(model, dataset(evaluation_records['test_within']), source_records=evaluation_records['test_within'], training_records=train_records, device='cuda', memory_mode='adaptive', max_tokens=131_072))
result = {'candidate_id':'titans-memory','experiment':experiment,'parameters':model.parameter_count(),'recurrent_state_elements_per_stream':model.recurrent_state_elements(),'training':{'steps':trainer.state.step,'tokens':trainer.state.tokens_seen,'wall_time_seconds':wall_time,'tokens_per_second':trainer.state.tokens_seen/wall_time,'peak_memory_bytes':training_peak,'gpu_utilization':gpu_sampler.summary()},'evaluation':evaluation,'stream_evaluation':stream_evaluation,'repeat_context':repeat_context,'checkpoint':str(checkpoint)}
(run_dir / 'result.json').write_text(json.dumps(result, indent=2, sort_keys=True) + '\n')
print(json.dumps(result, indent=2))
